In [1]:
import time
import numpy as np
import xesmf as xe
import xarray as xr
from pathlib import Path
from dask.distributed import Client

import warnings
warnings.filterwarnings("ignore")

In [2]:
c = Client(threads_per_worker=1)

2026-04-30 17:29:11,454 - distributed.nanny.memory - WARNING - Worker tcp://127.0.0.1:36719 (pid=14952) exceeded 95% memory budget. Restarting...
2026-04-30 17:29:11,521 - distributed.nanny - WARNING - Restarting worker
2026-04-30 17:29:14,602 - distributed.nanny.memory - WARNING - Worker tcp://127.0.0.1:37623 (pid=14955) exceeded 95% memory budget. Restarting...
2026-04-30 17:29:14,664 - distributed.scheduler - WARNING - Removing worker 'tcp://127.0.0.1:37623' caused the cluster to lose already computed task(s), which will be recomputed elsewhere: {'original-array-110885c26fad0a45d0a638b47d9cd701'} (stimulus_id='handle-worker-cleanup-1777534154.6642792')
2026-04-30 17:29:14,672 - distributed.nanny - WARNING - Restarting worker
2026-04-30 17:29:17,347 - distributed.nanny.memory - WARNING - Worker tcp://127.0.0.1:42389 (pid=14981) exceeded 95% memory budget. Restarting...
2026-04-30 17:29:17,380 - distributed.scheduler - WARNING - Removing worker 'tcp://127.0.0.1:42389' caused the clust

In [3]:
weights_base = Path("/g/data/nm03/lxy581/global_drag_coeff/")
weights_h2u = weights_base / "8km_h_to_8km_u_weights.nc"
weights_h2v = weights_base / "8km_h_to_8km_v_weights.nc"

In [4]:
output_base = Path("/scratch/nm03/lxy581/mom6/archive/tides_008_global_sigma_SAL_2layer_x02/output013")
output_interior = output_base / 'ocean_interior.nc'
output_static = output_base / "ocean_static.nc"

In [5]:
data = xr.open_dataset(output_interior, chunks={"time": 10, "zi": 1})
static = xr.open_dataset(output_static)

In [6]:
grid_h = xr.Dataset({"lat": static.geolat, "lon": static.geolon})
grid_u = xr.Dataset({"lat": static.geolat_u, "lon": static.geolon_u})

In [7]:
regridder_h2u = xe.Regridder(grid_h, grid_u, "bilinear", extrap_method="inverse_dist", reuse_weights=True, filename=weights_h2u)

In [8]:
def interp_h2u(var, weights_h2u):
    ds_h = xr.Dataset(data_vars={'var_h2u': var,
                                })
    ds_u = regridder_h2u(ds_h)
    return ds_u['var_h2u']

In [9]:
start_time = time.time()

In [10]:
e_h = data.e.squeeze()
e_u = interp_h2u(e_h, weights_h2u)

In [11]:
e_u = e_u.chunk({"time": 10})
eu_data = xr.Dataset({"e_u": e_u})
eu_data.to_netcdf("/g/data/nm03/lxy581/evaluate/amp_phase/e_h2u_8km.nc")

2026-04-30 17:29:11,511 - distributed.worker - ERROR - Worker stream died during communication: tcp://127.0.0.1:36719
Traceback (most recent call last):
  File "/g/data/xp65/public/apps/med_conda/envs/analysis3-26.03/lib/python3.12/site-packages/distributed/comm/tcp.py", line 228, in read
    frames_nosplit = await read_bytes_rw(stream, frames_nosplit_nbytes)
                     ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/g/data/xp65/public/apps/med_conda/envs/analysis3-26.03/lib/python3.12/site-packages/distributed/comm/tcp.py", line 367, in read_bytes_rw
    actual = await stream.read_into(chunk)  # type: ignore[arg-type]
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
tornado.iostream.StreamClosedError: Stream is closed

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "/g/data/xp65/public/apps/med_conda/envs/analysis3-26.03/lib/python3.12/site-packages/distributed/worker.py", line 2081, in gather_dep
    res

KilledWorker: Attempted to run task ('sum-sum-aggregate-store-map-3bfe14c5649a12d2fbcf1131e0671ffc', 9, 2, 0, 0) on 4 different workers, but all those workers died while running it. The last worker that attempt to run the task was tcp://127.0.0.1:33471. Inspecting worker logs is often a good next step to diagnose what went wrong. For more information see https://distributed.dask.org/en/stable/killed.html.

In [ ]:
end_time = time.time()
exe_time = float(end_time - start_time)
print("Execution time: %.1f seconds! \n" % exe_time)